# UC Xenium preprocessing

Turns the raw Single-Cell-Portal exports in `UC_Xenium_Data_Raw/` into one
`adata.h5ad` per sample, laid out to mirror the existing `Xenium/` dataset folder:

```
UC_Xenium/
  Sample_1_UC1_inflamed/adata.h5ad        # inflamed
  Sample_2_UC1_less_inflamed/adata.h5ad   # less inflamed
```

Each adata holds **raw counts in `.X`** (no normalized layer — SpaceTravLR and
harreman build their own), `.obs['cell_type']`, `.obsm['spatial']`, and the
dataset's precomputed `.obsm['X_umap']` (so SpaceTravLR's `setup_` skips
recomputing PCA/neighbors/UMAP).


In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import os
import pandas as pd
import anndata as ad

from metab_processing.metab_travlr_config import DATA_DIR

In [ ]:
# Raw exports live next to the Xenium folder; adata objects go in UC_Xenium/.
RAW_DIR = f'{DATA_DIR}/UC_Xenium_Data_Raw'
OUT_DIR = f'{DATA_DIR}/UC_Xenium'

# folder name -> (biosample_id in metadata, its spatial-coordinate file)
SAMPLES = {
    'Sample_1_UC1_inflamed':      ('UC1_inflamed',      'spatial_UC1_1.tsv'),
    'Sample_2_UC1_less_inflamed': ('UC1_less_inflamed', 'spatial_UC1_2.tsv'),
}

In [ ]:
# Shared inputs (all cells from both samples).
# The .tsv files have a second 'TYPE' header row we skip; NAME is the global cell index.
meta = pd.read_csv(f'{RAW_DIR}/spatial_metadata.tsv', sep='\t', skiprows=[1], index_col=0)

# Precomputed UMAP provided with the dataset (all cells); saved so SpaceTravLR
# skips recomputing PCA/neighbors/UMAP on these large samples.
umap = pd.read_csv(f'{RAW_DIR}/spatial_cluster_umap.tsv', sep='\t', skiprows=[1], index_col=0)

# Dense counts are genes x cells and gzip-compressed despite the .tsv name.
# Read counts as int16 (Xenium counts are tiny) to keep this ~871k-cell matrix small in memory.
counts_path = f'{RAW_DIR}/spatial_raw_counts_dense.tsv'
cell_cols = pd.read_csv(counts_path, sep='\t', nrows=0, compression='gzip').columns
expr = pd.read_csv(counts_path, sep='\t', index_col=0, compression='gzip',
                   dtype={c: 'int16' for c in cell_cols if c != 'GENE'})
expr.columns = expr.columns.astype(int)   # cell indices as ints, to match metadata/coords

print('metadata:', meta.shape, '| counts (genes x cells):', expr.shape)

In [ ]:
for folder, (biosample, coord_file) in SAMPLES.items():
    cells = meta.index[meta['biosample_id'] == biosample]
    coords = pd.read_csv(f'{RAW_DIR}/{coord_file}', sep='\t', skiprows=[1], index_col=0)

    obs = meta.loc[cells].copy()
    obs['cell_type'] = obs['cell_types']

    adata = ad.AnnData(
        X=expr[cells].T.values.astype('float32'),   # cells x genes, raw counts
        obs=obs,
        var=pd.DataFrame(index=expr.index),
    )
    adata.obs_names = cells.astype(str)
    adata.var_names_make_unique()
    adata.obsm['spatial'] = coords.loc[cells, ['X', 'Y']].to_numpy()
    adata.obsm['X_umap'] = umap.loc[cells, ['X', 'Y']].to_numpy()

    out = f'{OUT_DIR}/{folder}'
    os.makedirs(out, exist_ok=True)
    adata.write_h5ad(f'{out}/adata.h5ad')
    print(folder, '->', adata.shape)
    print(adata)